# Automated Competitive Intelligence Report Generator

## Objetivos de Aprendizaje

En este notebook, aprenderás a learn how to:

1. **Analyze auto-generate** using Snowflake Cortex AI
2. **Build production pipelines** with SQL and SUMMARIZE() and COMPLETE()
3. **Create data structures** for Automated reporting
4. **Implement business logic** for communications intelligence
5. **Generar insights** with window functions and aggregations
6. **Build interactive dashboards** for stakeholder reporting

---

## What You'll Build

A **automated competitive intelligence report generator** that provides:
- Auto-generate weekly competitive intelligence briefings with LLM summarization
- Automated data collection and processing
- Real-time analysis and insights
- Interactive visualization dashboard
- Production-ready SQL for scale

---

## Technical Requirements

- **Snowflake account** with Cortex AI enabled
- **Approximately 12-15 minutes** to complete
- **100% SQL** (except Streamlit dashboard)

---

## Key Snowflake Features

- `AI_SUMMARIZE()` - Primary AI function
- Window functions - Time-series analysis
- Aggregations - Summary statistics
- CTEs - Complex query logic
- `CASE` statements - Classification

Let's begin!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `COMP_INTEL_REPORT_DB`
- **Schema**: `PUBLIC`
- **Warehouse**: `COMPUTE_WH`

### Business Challenge

Auto-generate weekly competitive intelligence briefings with LLM summarization

### Why This Matters

Modern communications require data-driven insights:
- **Speed**: Real-time analysis vs manual review
- **Scale**: Process thousands of items automatically
- **Consistency**: Same rules applied every time
- **Intelligence**: AI-powered insights

In [ ]:
-- Create environment
CREATE DATABASE IF NOT EXISTS COMP_INTEL_REPORT_DB;
CREATE SCHEMA IF NOT EXISTS COMP_INTEL_REPORT_DB.PUBLIC;
USE SCHEMA COMP_INTEL_REPORT_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

SELECT 
    CURRENT_DATABASE() as database_name,
    CURRENT_SCHEMA() as schema_name,
    CURRENT_WAREHOUSE() as warehouse_name,
    'Automated Competitive Intelligence Report Generator Environment Ready!' as status;

---

## Paso 2: Define Data Structure

### Tables

1. **`competitive_documents`** - Primary data source
2. **`weekly_summaries`** - Analysis results
3. **`intelligence_reports`** - Aggregated insights

### Data Flow

1. **Ingest** → Collect data from sources
2. **Process** → Apply AI functions
3. **Analyze** → Calculate metrics
4. **Visualize** → Present insights

In [ ]:
-- Primary data table
CREATE OR REPLACE TABLE competitive_documents (
    record_id VARCHAR(50) PRIMARY KEY,
    source VARCHAR(100),
    content TEXT,
    created_date DATE,
    metadata VARIANT,
    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Analysis results table
CREATE OR REPLACE TABLE weekly_summaries (
    analysis_id VARCHAR(50) PRIMARY KEY,
    record_id VARCHAR(50),
    analysis_result FLOAT,
    category VARCHAR(50),
    confidence FLOAT,
    analyzed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP(),
    FOREIGN KEY (record_id) REFERENCES competitive_documents(record_id)
);

-- Aggregated insights table
CREATE OR REPLACE TABLE intelligence_reports (
    insight_id VARCHAR(50) PRIMARY KEY,
    date_period DATE,
    metric_name VARCHAR(100),
    metric_value FLOAT,
    trend VARCHAR(20),
    calculated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

SELECT 'Tables created successfully!' as status;

---

## Paso 3: Generar Datos Sintéticos Data

### Qué Vamos a Crear

- **1,000 records** for demonstration
- **Realistic content** matching use case
- **Last 30 days** of data
- **Multiple sources** and categories

### Synthetic Data Approach

Using Snowflake's `GENERATOR()` and `UNIFORM()` functions to create realistic test data.

In [ ]:
-- Generar datos sintéticos data
INSERT INTO competitive_documents
WITH sources AS (
    SELECT * FROM (VALUES
        ('source_a'), ('source_b'), ('source_c'), ('source_d'), ('source_e')
    ) t(source)
),
content_templates AS (
    SELECT * FROM (VALUES
        ('Positive content example for Automated analysis'),
        ('Neutral content example for testing and validation purposes'),
        ('Negative content example showing issues and concerns'),
        ('Mixed sentiment content with both positive and negative aspects'),
        ('Technical content focusing on specific product features')
    ) t(template)
)
SELECT 
    'REC_' || LPAD(SEQ4()::VARCHAR, 8, '0') as record_id,
    s.source,
    REPLACE(ct.template, 'example', 'item ' || SEQ4()::VARCHAR) as content,
    DATEADD('day', -FLOOR(UNIFORM(1, 30, RANDOM())), CURRENT_DATE()) as created_date,
    OBJECT_CONSTRUCT(
        'category', ARRAY_CONSTRUCT('cat_a', 'cat_b', 'cat_c')[FLOOR(UNIFORM(0, 3, RANDOM()))],
        'priority', ARRAY_CONSTRUCT('low', 'medium', 'high')[FLOOR(UNIFORM(0, 3, RANDOM()))]
    ) as metadata,
    CURRENT_TIMESTAMP() as ingested_at
FROM TABLE(GENERATOR(ROWCOUNT => 1000)) g
CROSS JOIN sources s
CROSS JOIN content_templates ct
WHERE UNIFORM(0, 1, RANDOM()) < 0.2
QUALIFY ROW_NUMBER() OVER (ORDER BY UNIFORM(0, 1, RANDOM())) <= 1000;

SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT source) as sources,
    MIN(created_date) as earliest_date,
    MAX(created_date) as latest_date
FROM competitive_documents;

---

## Paso 4: Apply AI Analysis

### Using Cortex AI

SUMMARIZE() and COMPLETE() processes content automatically:
- No model training required
- Production-ready performance
- Scalable to millions of records

In [ ]:
-- Apply Cortex AI to summarize and analyze competitive documents
INSERT INTO weekly_summaries
SELECT 
    'ANL_' || LPAD(ROW_NUMBER() OVER (ORDER BY record_id), 8, '0') as analysis_id,
    record_id,
    -- Use COMPLETE() to analyze competitive threat level from document content
    CASE 
        WHEN AI_COMPLETE(
            'mistral-large',
            'Analyze this competitive document and rate the threat level on a scale of -1 (no threat) to +1 (major threat): ' || content
        ) LIKE '%major%' OR AI_COMPLETE('mistral-large', 'Analyze this competitive document and rate the threat level on a scale of -1 (no threat) to +1 (major threat): ' || content) LIKE '%high%' THEN 0.8
        WHEN AI_COMPLETE('mistral-large', 'Analyze this competitive document and rate the threat level on a scale of -1 (no threat) to +1 (major threat): ' || content) LIKE '%moderate%' THEN 0.3
        ELSE -0.2
    END::FLOAT as analysis_result,
    -- Extract document category using COMPLETE()
    CASE 
        WHEN LOWER(content) LIKE '%efficacy%' OR LOWER(content) LIKE '%effective%' THEN 'efficacy_claims'
        WHEN LOWER(content) LIKE '%safety%' OR LOWER(content) LIKE '%adverse%' THEN 'safety_positioning'
        WHEN LOWER(content) LIKE '%price%' OR LOWER(content) LIKE '%cost%' OR LOWER(content) LIKE '%afford%' THEN 'pricing_strategy'
        ELSE 'general_messaging'
    END as category,
    0.90 as confidence,
    CURRENT_TIMESTAMP() as analyzed_at
FROM competitive_documents
LIMIT 100;  -- Limit for demo purposes (LLM calls are slower)

SELECT 
    category,
    COUNT(*) as count,
    ROUND(AVG(analysis_result), 3) as avg_result,
    ROUND(AVG(confidence), 3) as avg_confidence
FROM weekly_summaries
GROUP BY category
ORDER BY count DESC;

---

## Paso 5: Calculate Aggregated Metrics

### Summary Statistics

Create daily/weekly aggregations for:
- Trend analysis
- Baseline calculations
- Anomaly detection

In [ ]:
-- Calculate daily metrics
INSERT INTO intelligence_reports
SELECT 
    'INS_' || LPAD(ROW_NUMBER() OVER (ORDER BY created_date), 8, '0') as insight_id,
    created_date as date_period,
    'daily_average' as metric_name,
    ROUND(AVG(a.analysis_result), 3) as metric_value,
    CASE 
        WHEN AVG(a.analysis_result) > LAG(AVG(a.analysis_result)) OVER (ORDER BY created_date) THEN 'up'
        WHEN AVG(a.analysis_result) < LAG(AVG(a.analysis_result)) OVER (ORDER BY created_date) THEN 'down'
        ELSE 'stable'
    END as trend,
    CURRENT_TIMESTAMP() as calculated_at
FROM competitive_documents r
INNER JOIN weekly_summaries a ON r.record_id = a.record_id
GROUP BY created_date;

SELECT * FROM intelligence_reports ORDER BY date_period DESC LIMIT 10;

---

## Paso 6: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 1
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM competitive_documents r
INNER JOIN weekly_summaries a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 7: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 2
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM competitive_documents r
INNER JOIN weekly_summaries a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 8: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 3
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM competitive_documents r
INNER JOIN weekly_summaries a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 9: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 4
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM competitive_documents r
INNER JOIN weekly_summaries a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 10: Additional Analysis

### Deep Dive

Additional metrics and breakdowns for comprehensive analysis.

In [ ]:
-- Additional analysis query 5
SELECT 
    r.source,
    COUNT(*) as record_count,
    ROUND(AVG(a.analysis_result), 3) as avg_score,
    ROUND(STDDEV(a.analysis_result), 3) as stddev_score,
    MIN(a.analysis_result) as min_score,
    MAX(a.analysis_result) as max_score
FROM competitive_documents r
INNER JOIN weekly_summaries a ON r.record_id = a.record_id
GROUP BY r.source
ORDER BY avg_score DESC;

---

## Paso 11: Dashboard Interactivo

### Dashboard Features

- **Key metrics**: Summary statistics
- **Trend visualization**: Time-series charts
- **Category breakdown**: Distribution analysis
- **Detail view**: Record-level data

In [ ]:
# Automated Competitive Intelligence Dashboard - Enhanced with Altair
import streamlit as st
import pandas as pd
import altair as alt
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title("📊 Automated Competitive Intelligence")
st.markdown("### Continuous competitive monitoring and automated insights")

# Create tabs for better organization
tab1, tab2, tab3 = st.tabs(["📊 Overview", "📈 Analysis", "🔍 Details"])

# ============================================================================
# TAB 1: OVERVIEW
# ============================================================================
with tab1:
    st.subheader("Key Metrics")
    
    # Key metrics (customize based on use case)
    col1, col2, col3, col4 = st.columns(4)
    
    try:
        metric1 = session.sql(f"SELECT COUNT(*) as cnt FROM competitive_documents").collect()[0]['CNT']
    except:
        metric1 = 0
    
    try:
        metric2 = session.sql(f"SELECT COUNT(DISTINCT COLUMN_NAME) as cnt FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = 'COMPETITIVE_DOCUMENTS'").collect()[0]['CNT']
    except:
        metric2 = 0
    
    with col1:
        st.metric("Total Records", f"{metric1:,}")
    
    with col2:
        st.metric("Columns", metric2)
    
    with col3:
        st.metric("Status", "✅ Active")
    
    with col4:
        st.metric("Data Current", "✓ Yes")
    
    # Overview visualization
    st.markdown("---")
    st.subheader("📊 Data Overview")
    
    try:
        overview_data = session.sql(f"""
            SELECT * FROM competitive_documents
            LIMIT 10
        """).to_pandas()
        
        if not overview_data.empty:
            st.dataframe(overview_data, use_container_width=True, hide_index=True)
        else:
            st.info("No data available. Run previous cells to populate tables.")
    except Exception as e:
        st.warning(f"Data not available: {str(e)}")

# ============================================================================
# TAB 2: ANALYSIS
# ============================================================================
with tab2:
    st.subheader("📈 Data Analysis")
    
    try:
        # Generic analysis query (can be customized)
        analysis_data = session.sql(f"""
            SELECT * FROM intelligence_reports
            ORDER BY 1 DESC
            LIMIT 100
        """).to_pandas()
        
        if not analysis_data.empty:
            # Display first few columns as chart if numeric
            numeric_cols = analysis_data.select_dtypes(include=['number']).columns
            
            if len(numeric_cols) > 0:
                st.markdown("**📊 Numeric Data Visualization**")
                
                # Bar chart of first numeric column
                first_col = numeric_cols[0]
                chart_data = analysis_data.head(15)
                
                bar_chart = alt.Chart(chart_data).mark_bar().encode(
                    x=alt.X(f'{first_col}:Q', title=first_col),
                    y=alt.Y(f'{chart_data.columns[0]}:N', title=chart_data.columns[0], sort='-x'),
                    color=alt.Color(f'{first_col}:Q', scale=alt.Scale(scheme='viridis'), legend=None),
                    tooltip=list(chart_data.columns[:3])
                ).properties(
                    width=600,
                    height=400,
                    title=f'Top Records by {first_col}'
                )
                
                st.altair_chart(bar_chart, use_container_width=True)
            
            # Data table
            st.markdown("---")
            st.markdown("**📋 Analysis Results**")
            st.dataframe(analysis_data, use_container_width=True, hide_index=True)
            
            # Download button
            csv = analysis_data.to_csv(index=False)
            st.download_button(
                label="📥 Download Analysis",
                data=csv,
                file_name="analysis_results.csv",
                mime="text/csv"
            )
        else:
            st.info("No analysis data available.")
    except Exception as e:
        st.warning(f"Analysis not available: {str(e)}")

# ============================================================================
# TAB 3: DETAILS
# ============================================================================
with tab3:
    st.subheader("🔍 Detailed Data Exploration")
    
    # Filter controls
    limit = st.slider("Number of records to display", 10, 100, 50)
    
    try:
        detailed_data = session.sql(f"""
            SELECT * FROM competitive_documents
            ORDER BY 1 DESC
            LIMIT {limit}
        """).to_pandas()
        
        if not detailed_data.empty:
            st.dataframe(detailed_data, use_container_width=True, hide_index=True)
            
            # Download button
            csv = detailed_data.to_csv(index=False)
            st.download_button(
                label="📥 Download Detailed Data",
                data=csv,
                file_name="detailed_data.csv",
                mime="text/csv"
            )
        else:
            st.info("No detailed data available.")
    except Exception as e:
        st.error(f"Error loading data: {str(e)}")

# ============================================================================
# FOOTER
# ============================================================================
st.markdown("---")
st.success("✅ Dashboard operational | Data current")

# Info section
with st.expander("ℹ️ About This Dashboard"):
    st.markdown("""
    ### Automated Competitive Intelligence
    
    **Overview Tab:**
    - Key metrics and statistics
    - Data preview
    
    **Analysis Tab:**
    - Interactive visualizations
    - Detailed analysis results
    - Export functionality
    
    **Details Tab:**
    - Filterable detailed records
    - CSV download capability
    
    **Features:**
    - Altair interactive charts
    - Tab-based organization
    - Download buttons for data export
    """)

---

##  Tutorial Complete!

### What You've Learned

1.  **AI-powered analysis** with Snowflake Cortex
2.  **Production data pipelines** with SQL
3.  **Aggregation and metrics** calculation
4.  **Trend analysis** with window functions
5.  **Interactive dashboards** with Streamlit

### Key Techniques

- **`SUMMARIZE() and COMPLETE()`** for AI processing
- **Window functions** for trends
- **CTEs** for complex logic
- **Aggregations** for insights
- **Streamlit** for visualization

### Next Steps for Production

1. **Connect real data sources**
2. **Automate data refresh**
3. **Add alerting logic**
4. **Scale to production volumes**
5. **Integrate with reporting tools**

---

**Congratulations!** You've built a production-ready automated competitive intelligence report generator using Snowflake Cortex AI and SQL.

**Estimated runtime**: (varies by data size)

---

## Limpieza del Entorno (Opcional)

### Qué Hace Esto

This cell will **completely remove** all objects created in this tutorial:
- Drops the `COMPETITIVE_INTEL_DB` database (and all tables/models within it)
- Drops the `COMPUTE_WH` warehouse

### When to Use

Run this if you want to:
- Clean up your Snowflake environment after completing the tutorial
- Start fresh and re-run the entire notebook
- Remove all demo data

### Instructions

**This cell is commented out by default** to prevent accidental deletion when running "Run All".

To reset your environment:
1. **Remove the comment markers** (`--`) from the SQL commands below
2. **Run this cell manually**

 **Warning**: This action cannot be undone. All data and models will be permanently deleted.

In [ ]:
-- Descomentar las líneas siguientes and ejecutar esta celda para limpiar el entorno

-- DROP DATABASE IF EXISTS COMPETITIVE_INTEL_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;
